In [ ]:
#BB84 PROTOCOL CODE

import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

simulator = AerSimulator()
print("Ready.")

In [ ]:
def prepare_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc

def measure_qubit(qc, basis):
    qc = qc.copy()
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    return qc

def run_circuit(qc):
    job = simulator.run(qc, shots=1, memory=True)
    return int(job.result().get_memory()[0])

print("Functions redefined.")

In [ ]:
N = 100  # number of qubits

alice_bits  = [random.randint(0, 1) for _ in range(N)]
alice_bases = [random.randint(0, 1) for _ in range(N)]

circuits = [prepare_qubit(alice_bits[i], alice_bases[i]) for i in range(N)]

print(f"Alice prepared {N} qubits.")
print(f"First 10 bits:  {alice_bits[:10]}")
print(f"First 10 bases: {alice_bases[:10]}  (0=Z, 1=X)")

In [ ]:
bob_bases   = [random.randint(0, 1) for _ in range(N)]
bob_results = []

for i in range(N):
    qc     = measure_qubit(circuits[i], bob_bases[i])
    result = run_circuit(qc)
    bob_results.append(result)

print(f"First 10 Bob bases:   {bob_bases[:10]}  (0=Z, 1=X)")
print(f"First 10 Bob results: {bob_results[:10]}")

In [ ]:
matching_indices = [i for i in range(N) if alice_bases[i] == bob_bases[i]]

alice_sifted = [alice_bits[i]  for i in matching_indices]
bob_sifted   = [bob_results[i] for i in matching_indices]

print(f"Total qubits sent : {N}")
print(f"Matching bases    : {len(matching_indices)}  ({len(matching_indices)/N:.1%})")
print(f"Alice sifted key  : {alice_sifted[:20]}")
print(f"Bob   sifted key  : {bob_sifted[:20]}")
print(f"Keys match        : {alice_sifted == bob_sifted}")

In [ ]:
SAMPLE_FRACTION = 0.25
QBER_THRESHOLD  = 0.10

n_sample       = max(1, int(len(alice_sifted) * SAMPLE_FRACTION))
sample_indices = random.sample(range(len(alice_sifted)), n_sample)
errors         = sum(1 for i in sample_indices if alice_sifted[i] != bob_sifted[i])
qber           = errors / n_sample

print(f"Bits sampled for error check : {n_sample}")
print(f"Errors found                 : {errors}")
print(f"QBER                         : {qber:.2%}")

if qber < QBER_THRESHOLD:
    key_indices = [i for i in range(len(alice_sifted)) if i not in sample_indices]
    alice_key   = [alice_sifted[i] for i in key_indices]
    bob_key     = [bob_sifted[i]   for i in key_indices]
    print(f"\n Channel secure. Final key length: {len(alice_key)} bits")
    print(f"Alice key: {alice_key[:20]}...")
    print(f"Bob   key: {bob_key[:20]}...")
else:
    print(f"\n✗ QBER too high — eavesdropper detected, aborting.")
    alice_key = bob_key = []

In [ ]:
def run_bb84(n_qubits, eve_present=False):
    _alice_bits  = [random.randint(0,1) for _ in range(n_qubits)]
    _alice_bases = [random.randint(0,1) for _ in range(n_qubits)]
    _bob_bases   = [random.randint(0,1) for _ in range(n_qubits)]
    _bob_results = []

    for i in range(n_qubits):
        qc = prepare_qubit(_alice_bits[i], _alice_bases[i])

        if eve_present:
            eve_basis  = random.randint(0, 1)
            qc_eve     = measure_qubit(qc, eve_basis)
            eve_result = run_circuit(qc_eve)
            qc         = prepare_qubit(eve_result, eve_basis)  # Eve re-prepares

        qc = measure_qubit(qc, _bob_bases[i])
        _bob_results.append(run_circuit(qc))

    _matching   = [i for i in range(n_qubits) if _alice_bases[i] == _bob_bases[i]]
    _alice_sift = [_alice_bits[i]   for i in _matching]
    _bob_sift   = [_bob_results[i]  for i in _matching]
    _qber       = sum(a != b for a, b in zip(_alice_sift, _bob_sift)) / max(1, len(_alice_sift))

    return _alice_sift, _bob_sift, _qber

_, _, qber_no_eve = run_bb84(500, eve_present=False)
_, _, qber_eve    = run_bb84(500, eve_present=True)

print(f"QBER without Eve : {qber_no_eve:.2%}  (expected ~0%)")
print(f"QBER with Eve    : {qber_eve:.2%}  (expected ~25%)")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 5))

cases = [
    (0, 0, 'bit=0, Z-basis', r'$|0\rangle$'),
    (1, 0, 'bit=1, Z-basis', r'$|1\rangle$'),
    (0, 1, 'bit=0, X-basis', r'$|+\rangle$'),
    (1, 1, 'bit=1, X-basis', r'$|-\rangle$'),
]

for col, (bit, basis, title, state) in enumerate(cases):
    qc_prep = prepare_qubit(bit, basis)
    qc_prep.draw('mpl', ax=axes[0][col])
    axes[0][col].set_title(f'Alice: {title}\nPrepares {state}', fontsize=9)

    qc_full = prepare_qubit(bit, basis)
    qc_full = measure_qubit(qc_full, basis)
    qc_full.draw('mpl', ax=axes[1][col])
    axes[1][col].set_title(f'Bob measures in same basis\nResult always = {bit}', fontsize=9)

plt.suptitle("BB84 Encoding and Measurement Circuits", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('encoding_measurements.png', dpi=150, bbox_inches='tight')

plt.show()


In [ ]:
#Use faster function to speed up runtime, still takes ~1 minute to run
def run_bb84_fast(n_qubits, eve_present=False):
    _alice_bits  = [random.randint(0,1) for _ in range(n_qubits)]
    _alice_bases = [random.randint(0,1) for _ in range(n_qubits)]
    _bob_bases   = [random.randint(0,1) for _ in range(n_qubits)]

    circuits_to_run = []
    for i in range(n_qubits):
        qc = prepare_qubit(_alice_bits[i], _alice_bases[i])
        if eve_present:
            eve_basis  = random.randint(0,1)
            if eve_basis != _alice_bases[i]:
                eve_result = random.randint(0,1)  # wrong basis = random
            else:
                eve_result = _alice_bits[i]
            qc = prepare_qubit(eve_result, eve_basis)
        qc = measure_qubit(qc, _bob_bases[i])
        circuits_to_run.append(qc)

    job     = simulator.run(circuits_to_run, shots=1, memory=True)
    results = [int(job.result().get_memory(i)[0]) for i in range(n_qubits)]

    _matching   = [i for i in range(n_qubits) if _alice_bases[i] == _bob_bases[i]]
    _alice_sift = [_alice_bits[i] for i in _matching]
    _bob_sift   = [results[i]     for i in _matching]
    _qber       = sum(a != b for a,b in zip(_alice_sift, _bob_sift)) / max(1, len(_alice_sift))

    return _alice_sift, _bob_sift, _qber

N_TRIALS = 250

qbers_no_eve = []
qbers_eve    = []

for _ in range(N_TRIALS):
    _, _, q = run_bb84_fast(200, eve_present=False)
    qbers_no_eve.append(q)
    _, _, q = run_bb84_fast(200, eve_present=True)
    qbers_eve.append(q)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(qbers_no_eve, bins=20, alpha=0.7, color='steelblue', density=True, label='No Eve')
ax.hist(qbers_eve,    bins=20, alpha=0.7, color='crimson',   density=True, label='Eve present')
ax.axvline(0.10, color='black',   linestyle='--', linewidth=1.5, label='Abort threshold (10%)')
ax.axvline(0.25, color='crimson', linestyle=':',  linewidth=1.5, label='Expected QBER with Eve (25%)')
ax.set_xlabel('QBER', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title(f'QBER Distribution over {N_TRIALS} trials', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('qber_distribution.png', dpi=150, bbox_inches='tight')

plt.show()


print(f"Mean QBER without Eve : {np.mean(qbers_no_eve):.2%}")
print(f"Mean QBER with Eve    : {np.mean(qbers_eve):.2%}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def qber_vs_eve_rate(n_qubits, eve_rate):
    """
    eve_rate: fraction of qubits Eve intercepts (0.0 to 1.0)
    """
    alice_bits  = [random.randint(0,1) for _ in range(n_qubits)]
    alice_bases = [random.randint(0,1) for _ in range(n_qubits)]
    bob_bases   = [random.randint(0,1) for _ in range(n_qubits)]
    bob_results = []

    for i in range(n_qubits):
        if random.random() < eve_rate:
            # Eve intercepts this qubit
            eve_basis = random.randint(0,1)
            if eve_basis == alice_bases[i]:
                sent_bit   = alice_bits[i]
                sent_basis = eve_basis
            else:
                sent_bit   = random.randint(0,1)
                sent_basis = eve_basis
        else:
            sent_bit   = alice_bits[i]
            sent_basis = alice_bases[i]

        if bob_bases[i] == sent_basis:
            bob_results.append(sent_bit)
        else:
            bob_results.append(random.randint(0,1))

    matching   = [i for i in range(n_qubits) if alice_bases[i] == bob_bases[i]]
    alice_sift = [alice_bits[i]  for i in matching]
    bob_sift   = [bob_results[i] for i in matching]
    qber       = sum(a != b for a,b in zip(alice_sift, bob_sift)) / max(1, len(alice_sift))

    return qber

eve_rates    = np.linspace(0, 1, 30)
N_QUBITS     = 500
N_REPEATS    = 50

mean_qbers = []
std_qbers  = []

for rate in eve_rates:
    qbers = [qber_vs_eve_rate(N_QUBITS, rate) for _ in range(N_REPEATS)]
    mean_qbers.append(np.mean(qbers))
    std_qbers.append(np.std(qbers))

mean_qbers = np.array(mean_qbers)
std_qbers  = np.array(std_qbers)
theoretical = 0.25 * eve_rates  

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(eve_rates * 100, mean_qbers * 100,
        color='steelblue', linewidth=2, label='Simulated QBER')
ax.fill_between(eve_rates * 100,
                (mean_qbers - std_qbers) * 100,
                (mean_qbers + std_qbers) * 100,
                alpha=0.2, color='steelblue', label='±1 std dev')
ax.plot(eve_rates * 100, theoretical * 100,
        color='gray', linestyle='--', linewidth=1.5, label='Theoretical (0.25 × interception rate)')
ax.axhline(10, color='crimson', linestyle='--', linewidth=1.5, label='Abort threshold (10%)')

crossing = 10 / 25 * 100  
ax.axvline(crossing, color='crimson', linestyle=':', linewidth=1)
ax.annotate(f'Eve must intercept\n<{crossing:.0f}% to stay\nbelow threshold',
            xy=(crossing, 10), xytext=(crossing + 8, 6),
            fontsize=8, color='crimson',
            arrowprops=dict(arrowstyle='->', color='crimson'))

ax.set_xlabel('Fraction of qubits intercepted by Eve (%)', fontsize=12)
ax.set_ylabel('QBER (%)', fontsize=12)
ax.set_title('QBER vs Eve Interception Rate', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(0, 100)
ax.set_ylim(0, 30)

plt.tight_layout()
fig.savefig('qber_vs_eve_rate.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Eve must intercept more than {crossing:.0f}% of qubits to push QBER above threshold.")

In [ ]:
n_bits = np.arange(1, 51)
p_catch = (1 - 0.75**n_bits) * 100

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(n_bits, p_catch,
        color='steelblue', linewidth=1, alpha=0.4, zorder=1)
ax.scatter(n_bits, p_catch,
           color='steelblue', s=25, zorder=2)

ax.axhline(95, color='crimson', linestyle='--', linewidth=1.2, label='95% confidence')
ax.axhline(99, color='darkred', linestyle='--', linewidth=1.2, label='99% confidence')

# Find where we cross each threshold
n_95 = np.argmax(p_catch >= 95) + 1
n_99 = np.argmax(p_catch >= 99) + 1

ax.axvline(n_95, color='crimson', linestyle=':', linewidth=1)
ax.axvline(n_99, color='darkred', linestyle=':', linewidth=1)

ax.annotate(f'n = {n_95}', xy=(n_95, 95), xytext=(n_95 + 1.5, 88),
            fontsize=9, color='crimson',
            arrowprops=dict(arrowstyle='->', color='crimson'))
ax.annotate(f'n = {n_99}', xy=(n_99, 99), xytext=(n_99 + 1.5, 92),
            fontsize=9, color='darkred',
            arrowprops=dict(arrowstyle='->', color='darkred'))

ax.set_xlabel('Number of bits sacrificed for error checking (n)', fontsize=12)
ax.set_ylabel('Probability of catching Eve (%)', fontsize=12)
ax.set_title(r'Eve Detection Probability: $1 - (0.75)^n$', fontsize=13)
ax.set_xlim(1, 50)
ax.set_ylim(0, 100)
ax.legend(fontsize=10)

plt.tight_layout()
fig.savefig('eve_detection_probability.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Need {n_95} bits to catch Eve with 95% confidence")
print(f"Need {n_99} bits to catch Eve with 99% confidence")

In [ ]:
#E91 PROTOCOL CODE BELOW

import random
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

simulator = AerSimulator()
print("Ready.")

In [ ]:
def make_singlet():
    """
    Prepares |ψ⁻⟩ = (1/√2)(|01⟩ - |10⟩)
    Qubit 0 → Alice, Qubit 1 → Bob
    """
    qc = QuantumCircuit(2, 2)
    qc.x(1)
    qc.h(1)
    qc.cx(1, 0)
    qc.x(0)        
    return qc

# Verify with statevector
from qiskit.quantum_info import Statevector

sv = Statevector(make_singlet())
print("Statevector:", sv)
print("\nExpected: 0.707|01⟩ - 0.707|10⟩")
make_singlet().draw('mpl')

In [ ]:
def measure_at_angle(qc, qubit, angle_deg, classical_bit):
    """
    Measure `qubit` along azimuthal angle `angle_deg` in the x-y plane.
    Applies Rz(-angle) then H to rotate into the correct basis before measuring.
    """
    angle_rad = np.deg2rad(angle_deg)
    qc.rz(-angle_rad, qubit)
    qc.h(qubit)
    qc.measure(qubit, classical_bit)
    return qc
    
ALICE_ANGLES = [0, 45, 90]
BOB_ANGLES   = [45, 90, 135]

print("Alice's measurement angles:", ALICE_ANGLES)
print("Bob's measurement angles:  ", BOB_ANGLES)

In [ ]:
def run_pair(alice_angle, bob_angle):
    qc = make_singlet()
    measure_at_angle(qc, 0, alice_angle, 0)  # Alice measures qubit 0
    measure_at_angle(qc, 1, bob_angle, 1)    # Bob measures qubit 1
    
    job    = simulator.run(qc, shots=1, memory=True)
    memory = job.result().get_memory()[0]
    
    alice_bit = int(memory[-1])
    bob_bit   = int(memory[-2])
    # convert 0/1 to +1/-1
    alice_result = 1 - 2 * alice_bit
    bob_result   = 1 - 2 * bob_bit
    
    return alice_result, bob_result


N_TEST = 200
results = [run_pair(45, 45) for _ in range(N_TEST)]
products = [a * b for a, b in results]
print(f"E(45°, 45°) = {np.mean(products):.3f}  (expected -1.000)")

results = [run_pair(0, 90) for _ in range(N_TEST)]
products = [a * b for a, b in results]
print(f"E(0°, 90°)  = {np.mean(products):.3f}  (expected  0.000)")

results = [run_pair(0, 180) for _ in range(N_TEST)]
products = [a * b for a, b in results]
print(f"E(0°, 180°) = {np.mean(products):.3f}  (expected +1.000)")

In [ ]:
def compute_E(alice_angle, bob_angle, n_shots=1000):
    """
    Estimate E(a, b) = <AB> by averaging product of outcomes over n_shots.
    """
    products = []
    for _ in range(n_shots):
        a, b = run_pair(alice_angle, bob_angle)
        products.append(a * b)
    return np.mean(products)

ALICE_ANGLES = [0, 45, 90]
BOB_ANGLES   = [45, 90, 135]

print("Full correlation table:")
print(f"{'':>6} | {'b=45°':>8} | {'b=90°':>8} | {'b=135°':>8}")
print("-" * 38)

E = {}
for a in ALICE_ANGLES:
    row = f"a={a:>2}° |"
    for b in BOB_ANGLES:
        e = compute_E(a, b)
        E[(a,b)] = e
        row += f"  {e:>6.3f}  |"
    print(row)

print("\nTheoretical values (-cos(a-b)):")
print(f"{'':>6} | {'b=45°':>8} | {'b=90°':>8} | {'b=135°':>8}")
print("-" * 38)
for a in ALICE_ANGLES:
    row = f"a={a:>2}° |"
    for b in BOB_ANGLES:
        theory = -np.cos(np.deg2rad(a - b))
        row += f"  {theory:>6.3f}  |"
    print(row)

In [ ]:
S = E[(0,45)] - E[(0,135)] + E[(90,45)] + E[(90,135)]

S_theory = (
    -np.cos(np.deg2rad(0-45))
    -(-np.cos(np.deg2rad(0-135)))
    -np.cos(np.deg2rad(90-45))
    -np.cos(np.deg2rad(90-135))
)

print(f"Measured S : {S:.4f}")
print(f"Theoretical: {-2*np.sqrt(2):.4f}")
print(f"Classical bound: |S| <= 2")
print(f"|S| = {abs(S):.4f}")

In [ ]:
def compute_E_with_uncertainty(alice_angle, bob_angle, n_shots=1000):
    products = []
    for _ in range(n_shots):
        a, b = run_pair(alice_angle, bob_angle)
        products.append(a * b)
    mean = np.mean(products)
    stderr = np.std(products) / np.sqrt(n_shots)
    return mean, stderr

N_SHOTS = 1000

E11, dE11 = compute_E_with_uncertainty(0,  45,  N_SHOTS)
E13, dE13 = compute_E_with_uncertainty(0,  135, N_SHOTS)
E31, dE31 = compute_E_with_uncertainty(90, 45,  N_SHOTS)
E33, dE33 = compute_E_with_uncertainty(90, 135, N_SHOTS)

S_measured = E11 - E13 + E31 + E33

S_uncertainty = np.sqrt(dE11**2 + dE13**2 + dE31**2 + dE33**2)

S_theory = -2 * np.sqrt(2)

n_sigma = abs(S_measured - S_theory) / S_uncertainty

print(f"E(0°,  45°)  = {E11:.4f} ± {dE11:.4f}")
print(f"E(0°,  135°) = {E13:.4f} ± {dE13:.4f}")
print(f"E(90°, 45°)  = {E31:.4f} ± {dE31:.4f}")
print(f"E(90°, 135°) = {E33:.4f} ± {dE33:.4f}")
print()
print(f"S = {S_measured:.4f} ± {S_uncertainty:.4f}")
print(f"Theoretical S = -2√2 = {S_theory:.4f}")
print(f"Deviation from theory: {n_sigma:.2f} sigma")
print()
print(f"S = -2√2  within {n_sigma:.2f}σ uncertainty: {n_sigma < 2}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
ALICE_ANGLES = [0, 45, 90]
BOB_ANGLES = [45, 90, 135]
# sizes
nA = len(ALICE_ANGLES)
nB = len(BOB_ANGLES)

# allocate matrices correctly
E_matrix = np.zeros((nA, nB))
E_theory = np.zeros((nA, nB))

# fill matrices
for i, a in enumerate(ALICE_ANGLES):
    for j, b in enumerate(BOB_ANGLES):
        E_matrix[i, j] = compute_E(a, b, n_shots=500)
        E_theory[i, j] = -np.cos(np.deg2rad(a - b))

# plotting
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, matrix, title in zip(
    axes,
    [E_matrix, E_theory],
    ['Simulated E(aᵢ, bⱼ)', 'Theoretical E(aᵢ, bⱼ) = -cos(aᵢ - bⱼ)']
):

    im = ax.imshow(matrix, vmin=-1, vmax=1, cmap='RdBu')

    # FIX: ticks match actual dimensions
    ax.set_xticks(range(nB))
    ax.set_yticks(range(nA))

    ax.set_xticklabels([f'b={b}°' for b in BOB_ANGLES])
    ax.set_yticklabels([f'a={a}°' for a in ALICE_ANGLES])

    ax.set_xlabel("Bob's angle", fontsize=11)
    ax.set_ylabel("Alice's angle", fontsize=11)
    ax.set_title(title, fontsize=11)

    # annotate values
    for i in range(nA):
        for j in range(nB):
            ax.text(j, i, f'{matrix[i, j]:.3f}',
                    ha='center', va='center',
                    fontsize=10, color='black')

    # --- highlight cells safely ---
    bell_cells = [(0,0), (0,2), (2,0), (2,2)]
    key_cells  = [(1,0), (2,1)]

    for (i, j) in bell_cells:
        if i < nA and j < nB:
            ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                         fill=False, edgecolor='gold', linewidth=3, zorder=3))

    for (i, j) in key_cells:
        if i < nA and j < nB:
            ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                         fill=False, edgecolor='lime', linewidth=3, zorder=3))

    plt.colorbar(im, ax=ax, label='Correlation E')

# legend
legend_elements = [
    Patch(facecolor='none', edgecolor='gold', linewidth=3, label='Used for S (Bell test)'),
    Patch(facecolor='none', edgecolor='lime', linewidth=3, label='Used for key'),
]

fig.legend(handles=legend_elements,
           loc='lower center',
           ncol=2,
           fontsize=10,
           bbox_to_anchor=(0.5, -0.05))

fig.suptitle('Correlation Function E(aᵢ, bⱼ) — Simulated vs Theoretical', fontsize=13)

plt.tight_layout()
fig.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
ALICE_ANGLES = [0, 90]
BOB_ANGLES   = [45, 135]

import numpy as np
import random

def run_pair_with_eve_classical(alice_angle, bob_angle, eve_present=False):
    if not eve_present:
        # Standard Singlet correlation: E = -cos(theta_A - theta_B)
        delta = np.deg2rad(alice_angle - bob_angle)
        p_agree = (1 - np.cos(delta)) / 2 # Note: 1-cos for anti-correlation of singlet
        alice_res = random.choice([-1, 1])
        bob_res = alice_res if random.random() < p_agree else -alice_res
        return alice_res, bob_res
    
    else:
        # 1. Eve intercepts. She is effectively Bob now.
        eve_angle = random.choice(ALICE_ANGLES + BOB_ANGLES)
        delta_ae = np.deg2rad(alice_angle - eve_angle)
        
        # Probability Alice and Eve agree (Singlet logic)
        p_agree_ae = (1 - np.cos(delta_ae)) / 2
        alice_res = random.choice([-1, 1])
        eve_res = alice_res if random.random() < p_agree_ae else -alice_res

        # 2. Eve resends her result to Bob. 
        # This is now a simple projection of her state (eve_res) onto Bob's basis.
        # It's no longer "entangled" logic, it's "Polarizer" logic.
        delta_eb = np.deg2rad(eve_angle - bob_angle)
        
        # Standard Malus' Law/Projection: E = cos(delta)
        p_agree_eb = (1 + np.cos(delta_eb)) / 2
        
        if random.random() < p_agree_eb:
            bob_res = eve_res
        else:
            bob_res = -eve_res
            
        return alice_res, bob_res


def compute_E_eve(alice_angle, bob_angle, eve_present=False, n_shots=500):
    products = []
    for _ in range(n_shots):
        a, b = run_pair_with_eve_classical(alice_angle, bob_angle, eve_present)
        products.append(a * b)
    mean   = np.mean(products)
    stderr = np.std(products) / np.sqrt(n_shots)
    return mean, stderr


def compute_S_eve(eve_present=False, n_shots=500):
    E11, dE11 = compute_E_eve(0,  45,  eve_present, n_shots)
    E13, dE13 = compute_E_eve(0,  135, eve_present, n_shots)
    E31, dE31 = compute_E_eve(90, 45,  eve_present, n_shots)
    E33, dE33 = compute_E_eve(90, 135, eve_present, n_shots)
    S     = E11 - E13 + E31 + E33
    S_unc = np.sqrt(dE11**2 + dE13**2 + dE31**2 + dE33**2)
    return S, S_unc


S_no_eve, dS_no_eve = compute_S_eve(eve_present=False)
S_eve,    dS_eve    = compute_S_eve(eve_present=True)

print(f"Without Eve : S = {S_no_eve:.4f} ± {dS_no_eve:.4f}  (expected {-2*np.sqrt(2):.4f})")
print(f"With Eve    : S = {S_eve:.4f}    ± {dS_eve:.4f}")
print()
print(f"Classical bound  : |S| <= 2.000")
print(f"|S| without Eve  : {abs(S_no_eve):.4f}  -- violation: {abs(S_no_eve) > 2}")
print(f"|S| with Eve     : {abs(S_eve):.4f}  -- violation: {abs(S_eve) > 2}")

In [ ]:
ABORT_THRESHOLD = -2.5

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(eve_rates * 100, S_means, 'o-',
        color='steelblue', linewidth=1.5, markersize=4, label='Simulated S')
ax.fill_between(eve_rates * 100,
                S_means - S_stds,
                S_means + S_stds,
                alpha=0.2, color='steelblue', label='±1 std dev')
ax.axhline(-2*np.sqrt(2), color='crimson', linestyle='--',
           linewidth=1.5, label=r'Quantum bound $-2\sqrt{2} \approx -2.828$')
ax.axhline(-2.0, color='gray', linestyle=':',
           linewidth=1.0, label='Classical bound (S = -2.0)')
ax.axhline(ABORT_THRESHOLD, color='orange', linestyle='--',
           linewidth=1.5, label=f'Practical abort threshold (S = {ABORT_THRESHOLD})')

crossings = np.where(S_means > ABORT_THRESHOLD)[0]
if len(crossings) > 0:
    crossing_idx  = crossings[0]
    crossing_rate = eve_rates[crossing_idx] * 100
    ax.axvline(crossing_rate, color='orange', linestyle=':', linewidth=1.2)
    ax.annotate(f'Abort triggered\n~{crossing_rate:.0f}% interception',
                xy=(crossing_rate, ABORT_THRESHOLD),
                xytext=(min(crossing_rate + 5, 75), ABORT_THRESHOLD - 0.25),
                fontsize=9, color='orange',
                arrowprops=dict(arrowstyle='->', color='orange'))

ax.set_xlabel('Fraction of qubits intercepted by Eve (%)', fontsize=12)
ax.set_ylabel('S', fontsize=12)
ax.set_title('Bell parameter S vs Eve interception rate in E91\n'
             'Eve picks randomly from {0°, 45°, 90°, 135°}',
             fontsize=12)
ax.set_xlim(0, 100)
ax.legend(fontsize=10, loc='lower right')
plt.tight_layout()
fig.savefig('E91_S_vs_eve_rate.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ALICE_ANGLES = [0, 45, 90]
BOB_ANGLES = [45, 90, 135]
S_no_eve_val = None
S_eve_val    = None

for idx, (ax, eve) in enumerate(zip(axes, [False, True])):
    matrix = np.zeros((3, 3))
    for i, a in enumerate(ALICE_ANGLES):
        for j, b in enumerate(BOB_ANGLES):
            matrix[i, j], _ = compute_E_eve(a, b, eve_present=eve, n_shots=300)

    im = ax.imshow(matrix, vmin=-1, vmax=1, cmap='RdBu')

    ax.set_xticks([0, 1, 2])
    ax.set_yticks([0, 1, 2])
    ax.set_xticklabels([f'b={b}°' for b in BOB_ANGLES])
    ax.set_yticklabels([f'a={a}°' for a in ALICE_ANGLES])
    ax.set_xlabel("Bob's angle", fontsize=11)
    ax.set_ylabel("Alice's angle", fontsize=11)

    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{matrix[i,j]:.3f}',
                    ha='center', va='center', fontsize=10, color='black')

    for (i, j) in [(0,0), (0,2), (2,0), (2,2)]:
        ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                     fill=False, edgecolor='gold', linewidth=3))

    fig.colorbar(im, ax=ax, label='E(aᵢ, bⱼ)')

    S = matrix[0,0] - matrix[0,2] + matrix[2,0] + matrix[2,2]

    if not eve:
        S_no_eve_val = S
    else:
        S_eve_val = S

axes[0].set_title(f'Without Eve\nS = {S_no_eve_val:.3f}  ' + r'($S \approx -2\sqrt{2}$)',
                  fontsize=11, color='green')
axes[1].set_title(f'With Eve\nS = {S_eve_val:.3f}  ' + r'($S \neq -2\sqrt{2}$)',
                  fontsize=11, color='crimson')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='none', edgecolor='gold',
                         linewidth=3, label='Used for S (Bell test)')]
fig.legend(handles=legend_elements, loc='lower center',
           fontsize=10, bbox_to_anchor=(0.5, -0.08))

fig.suptitle('Effect of Eve on Correlation Structure', fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig('E91_eve_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()